# ViTASA Enhanced — Pair Classification (Colab Free)

Run on Google Colab Free GPU (T4) with safe time limits.

**Formulation:** target-aspect pair classification (correct)
**Backbone:** ViSoBERT (pre-trained on social media)
**Metric:** macro F1 on 3 sentiment classes (loại 'none')

⏱️ **Thời gian an toàn cho Colab Free:**
- Baseline C1 đơn domain: ~15 phút ✅ SAFE
- Full ablation (4 configs × 3 domains × 10 epochs): ~2 tiếng ✅ SAFE
- Full ablation × 20 epochs: ~4 tiếng (có rủi ro timeout)

**Strategy:**
1. Chạy C1 (baseline) trước để xác nhận formulation đúng
2. Nếu OK → chạy C2-C4 từng domain 1 lần (tránh timeout)
3. Nếu timeout → giảm epochs hoặc chạy lại domain bị interrupt


In [ ]:
# 1. Clone dataset từ ViTASA repo gốc
!git clone https://github.com/kh4nh12/ViTASA.git ViTASA_repo 2>&1 | grep -E '(Cloning|clone|done)'
!echo "Dataset files:"
!ls -lh ViTASA_repo/*.jsonl

In [ ]:
# 2. Install dependencies
!pip install -q torch transformers scikit-learn seqeval underthesea
import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 3. Setup folder structure + copy dataset
import os
import shutil

os.makedirs('VITASA_Enhanced/baseline/data', exist_ok=True)
os.makedirs('VITASA_Enhanced/experiments/results_pair', exist_ok=True)

for domain in ['mobile', 'restaurant', 'hotel']:
    domain_dir = f'VITASA_Enhanced/baseline/data/{domain}'
    os.makedirs(domain_dir, exist_ok=True)
    src = f'ViTASA_repo/{domain}.jsonl'
    dst = f'{domain_dir}/{domain}.jsonl'
    if os.path.exists(src):
        shutil.copy(src, dst)
        lines = sum(1 for _ in open(dst))
        print(f"✅ {domain}: {lines} samples")

In [ ]:
# 4. Lấy code mới nhất từ GitHub (public repo — không cần Drive, không cần token)
# Mỗi lần bạn sửa code + git push, chỉ cần chạy lại CELL NÀY để lấy bản mới nhất,
# không phải kéo-thả file thủ công vào Drive nữa.
import os

REPO_URL = "https://github.com/Hunganh1305/VITASA_Enhanced.git"

if os.path.isdir("VITASA_Enhanced/.git"):
    !cd VITASA_Enhanced && git pull
else:
    !rm -rf VITASA_Enhanced_code_tmp
    !git clone {REPO_URL} VITASA_Enhanced_code_tmp
    # Merge vào folder VITASA_Enhanced (đã có sẵn baseline/data từ cell 3)
    !rsync -a VITASA_Enhanced_code_tmp/ VITASA_Enhanced/ --exclude 'baseline/data'
    !rm -rf VITASA_Enhanced_code_tmp

!echo "Files in VITASA_Enhanced:" && ls VITASA_Enhanced/ | grep -E '(train_pair|train_vitasd|text_norm|imbalanced)'

In [ ]:
# 4b. Mount Google Drive — BẮT BUỘC chạy cell này trước khi train.
# Đây là nguồn lưu trữ THẬT SỰ (persistent) — ổ đĩa local của Colab VM
# (`experiments/results_pair/`) sẽ mất sạch nếu runtime disconnect/hết quota.
# Cell train ở dưới dùng Drive làm "nguồn sự thật" duy nhất để biết domain/config
# nào đã xong — KHÔNG cần download qua popup nữa (hay bị trình duyệt chặn).

from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/VITASA_Enhanced_results_backup"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"✅ Kết quả sẽ tự backup vào: {DRIVE_BACKUP_DIR}")
print("   (Domain/config nào đã có results.json trong này sẽ được TỰ ĐỘNG SKIP,")
print("    dù bạn restart runtime hay mở lại notebook bất kỳ lúc nào.)")

In [ ]:
# 5. Smoke test: 1 epoch trên mobile
%cd VITASA_Enhanced

!python3 train_pair.py --domain mobile --loss ce --epochs 1 --subsample 0.1 --batch-size 64 --fp16 2>&1 | tail -30

print("\n✅ Smoke test passed!")

In [ ]:
# 6. STRATEGY 1: Run BASELINE ONLY (C1) — safest, ~45 min for 3 domains
# ✅ Resume-safe qua 2 nguồn — domain nào ĐÃ XONG ở nguồn nào cũng được nhận
#    diện và skip, kể cả đổi tài khoản Google/Colab giữa chừng:
#    (a) Drive của tài khoản ĐANG chạy (tự backup mỗi domain xong)
#    (b) results.json có sẵn trong repo sau `git pull` ở cell 4 — tức là nếu
#        bạn đã tải kết quả từ tài khoản KHÁC về máy rồi `git push` lên repo,
#        chỉ cần git pull lại là tài khoản mới này tự nhận ra, không train lại.
# ⚠️ Yêu cầu: đã chạy cell "4b. Mount Google Drive" ở trên.

import subprocess
import time
import shutil
from pathlib import Path

assert 'DRIVE_BACKUP_DIR' in dir(), (
    "❌ Chưa mount Google Drive! Chạy cell '4b. Mount Google Drive' ở trên trước."
)

LOCAL_RESULTS_DIR = Path("experiments/results_pair")  # có thể đã có sẵn nhờ git pull
DRIVE_DIR = Path(DRIVE_BACKUP_DIR) / "results_pair"
EPOCHS = 10
BATCH_SIZE = 64

for domain in ['mobile', 'restaurant', 'hotel']:
    config_name = f"pair_{domain}_loss-ce_phobert_mha"
    drive_result_file = DRIVE_DIR / config_name / "results.json"
    git_result_file = LOCAL_RESULTS_DIR / config_name / "results.json"

    print(f"\n{'='*70}")
    print(f"[{domain}] Baseline (C1) — EPOCHS={EPOCHS}")
    print(f"{'='*70}")

    if drive_result_file.exists():
        print(f"⏭️  Đã có trên Drive tài khoản này ({drive_result_file}) — bỏ qua.")
        continue
    if git_result_file.exists():
        print(f"⏭️  Đã có sẵn trong repo sau git pull ({git_result_file}) — bỏ qua, "
              f"chắc bạn đã chạy domain này ở tài khoản khác rồi push lên GitHub.")
        drive_result_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(git_result_file, drive_result_file)  # backup luôn vào Drive tài khoản này cho đồng bộ
        continue

    cmd = f"python3 train_pair.py --domain {domain} --loss ce --model phobert --epochs {EPOCHS} --batch-size {BATCH_SIZE} --fp16"
    result = subprocess.run(cmd.split(), capture_output=False)

    if result.returncode != 0:
        print(f"❌ ERROR: {domain} failed — KHÔNG có gì để backup, chạy lại cell này sau khi fix lỗi.")
        break

    if git_result_file.exists():
        drive_result_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(git_result_file, drive_result_file)
        print(f"💾 Backed up ngay lên Drive: {drive_result_file}")
    else:
        print(f"⚠️  {domain} chạy xong nhưng không thấy {git_result_file} — kiểm tra lại tên config.")

    time.sleep(5)

print("\n✅ Baseline runs complete! (Xem trực tiếp trong Drive:", DRIVE_DIR, ")")
print("📌 Muốn chuyển tiếp qua tài khoản khác: chạy cell 9 để tải kết quả về máy,")
print("   git add/commit/push, rồi mở Colab bằng tài khoản kia, git pull, chạy lại cell này.")

In [ ]:
# 7. STRATEGY 2: Full ablation — 4 configs × 3 domains
# ✅ Resume-safe qua 2 nguồn (Drive tài khoản hiện tại + git pull) — xem giải
#    thích chi tiết ở cell STRATEGY 1. Cho phép chạy nối tiếp qua nhiều tài
#    khoản Google khác nhau khi hết quota, miễn đã git push kết quả trước khi
#    đổi tài khoản.
# ⚠️ Yêu cầu: đã chạy cell "4b. Mount Google Drive" ở trên.

import subprocess
import time
import shutil
from pathlib import Path

assert 'DRIVE_BACKUP_DIR' in dir(), (
    "❌ Chưa mount Google Drive! Chạy cell '4b. Mount Google Drive' ở trên trước."
)

LOCAL_RESULTS_DIR = Path("experiments/results_pair")  # có thể đã có sẵn nhờ git pull
DRIVE_DIR = Path(DRIVE_BACKUP_DIR) / "results_pair"
EPOCHS = 10
BATCH_SIZE = 64

CONFIGS = [
    ("C1_baseline",   "--loss ce"),
    ("C2_norm",       "--loss ce --normalize"),
    ("C3_imbalanced", "--loss focal"),
    ("C4_full",       "--loss focal --normalize"),
]
DOMAINS = ['mobile', 'restaurant', 'hotel']

def config_name_for(domain, flags):
    norm_suffix = "_norm" if "--normalize" in flags else ""
    loss = "focal" if "focal" in flags else "ce"
    return f"pair_{domain}_loss-{loss}{norm_suffix}_phobert_mha"

print(f"Plan: {len(CONFIGS)} configs × {len(DOMAINS)} domains × {EPOCHS} epochs")
print(f"Estimated time: ~{len(CONFIGS)*len(DOMAINS)*EPOCHS//5} minutes (~{len(CONFIGS)*len(DOMAINS)*EPOCHS//5//60} hours)\n")

failed = []
for domain in DOMAINS:
    for config_label, flags in CONFIGS:
        config_name = config_name_for(domain, flags)
        drive_result_file = DRIVE_DIR / config_name / "results.json"
        git_result_file = LOCAL_RESULTS_DIR / config_name / "results.json"

        print(f"\n{'='*70}")
        print(f"[{domain}/{config_label}] — {EPOCHS} epochs")
        print(f"{'='*70}")

        if drive_result_file.exists():
            print(f"⏭️  Đã có trên Drive tài khoản này ({drive_result_file}) — bỏ qua.")
            continue
        if git_result_file.exists():
            print(f"⏭️  Đã có sẵn trong repo sau git pull ({git_result_file}) — bỏ qua "
                  f"(chạy ở tài khoản khác rồi push lên GitHub).")
            drive_result_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(git_result_file, drive_result_file)
            continue

        cmd = f"python3 train_pair.py --domain {domain} {flags} --model phobert --epochs {EPOCHS} --batch-size {BATCH_SIZE} --fp16"
        print(f"Command: {cmd}\n")

        result = subprocess.run(cmd.split(), capture_output=False)
        if result.returncode != 0:
            failed.append(f"{domain}/{config_label}")
            print(f"❌ ERROR: {domain}/{config_label} failed, skipping...")
            continue

        if git_result_file.exists():
            drive_result_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(git_result_file, drive_result_file)
            print(f"💾 Backed up ngay lên Drive: {drive_result_file}")
        else:
            print(f"⚠️  Chạy xong nhưng không thấy {git_result_file} — kiểm tra lại tên config.")

        time.sleep(3)

print(f"\n{'='*70}")
print(f"Ablation complete. Failed: {len(failed)}")
if failed:
    print(f"  {failed}")
print(f"{'='*70}")
print("Xem trực tiếp trong Drive:", DRIVE_DIR)
print("📌 Muốn chuyển tiếp qua tài khoản khác: chạy cell 9 để tải kết quả về máy,")
print("   git add/commit/push, rồi mở Colab bằng tài khoản kia, git pull, chạy lại cell này.")

In [ ]:
# 8. Print summary table — đọc từ Google Drive (nguồn thật, sống sót qua mọi session)
import json
from pathlib import Path
from collections import defaultdict

assert 'DRIVE_BACKUP_DIR' in dir(), (
    "❌ Chưa mount Google Drive! Chạy cell '4b. Mount Google Drive' ở trên trước."
)

results_dir = Path(DRIVE_BACKUP_DIR) / "results_pair"
results = defaultdict(dict)

BASELINE = {"mobile": 61.77, "restaurant": 41.12, "hotel": 52.64}

for results_file in sorted(results_dir.glob("*/results.json")):
    try:
        data = json.load(open(results_file))
        domain = data["domain"]
        config = data["config"]
        test_f1 = data["test"]["macro_f1"] * 100
        results[domain][config] = test_f1
    except Exception as e:
        print(f"[warn] {results_file}: {e}")

print("\n" + "="*90)
print("ABLATION RESULTS — macro F1 (3 sentiment classes, loại 'none')")
print("="*90)

header = f"{'Domain':<12} {'C1_baseline':>15} {'C2_norm':>15} {'C3_focal':>15} {'C4_full':>15} {'Baseline':>13}"
print(header)
print("-" * 90)

for domain in ['mobile', 'restaurant', 'hotel']:
    baseline = BASELINE[domain]
    c1_val = c2_val = c3_val = c4_val = None
    for config_key, f1_val in results.get(domain, {}).items():
        if 'loss-ce_phobert_mha' in config_key and 'norm' not in config_key:
            c1_val = f1_val
        elif 'loss-ce_norm' in config_key:
            c2_val = f1_val
        elif 'loss-focal_phobert_mha' in config_key and 'norm' not in config_key:
            c3_val = f1_val
        elif 'loss-focal_norm' in config_key:
            c4_val = f1_val

    c1_str = f"{c1_val:.2f}%" if c1_val else "—"
    c2_str = f"{c2_val:.2f}%" if c2_val else "—"
    c3_str = f"{c3_val:.2f}%" if c3_val else "—"
    c4_str = f"{c4_val:.2f}%" if c4_val else "—"

    print(f"{domain:<12} {c1_str:>15} {c2_str:>15} {c3_str:>15} {c4_str:>15} {baseline:>12.2f}%")

print("="*90)
print(f"\n✅ Nguồn: {results_dir}")
print(f"\n📥 Chạy cell tiếp theo để nén + tải toàn bộ kết quả từ Drive về máy.")

In [ ]:
# 9. Download results — nén trực tiếp từ Google Drive (đủ mọi domain/config đã
# backup qua các session trước, không phụ thuộc ổ đĩa tạm của session hiện tại)
from google.colab import files
from pathlib import Path

assert 'DRIVE_BACKUP_DIR' in dir(), (
    "❌ Chưa mount Google Drive! Chạy cell '4b. Mount Google Drive' ở trên trước."
)

results_dir = Path(DRIVE_BACKUP_DIR) / "results_pair"

!tar -czf VITASA_pair_results.tar.gz -C "{results_dir.parent}" results_pair
!ls -lh VITASA_pair_results.tar.gz

print("\n📥 Downloading results...")
files.download("VITASA_pair_results.tar.gz")
print("✅ Done!")

## Notes

**Nếu timeout:**
1. Colab Free có thể timeout sau 12h hoặc disconnect nếu idle 30 phút
2. Nếu bị interrupt → restart từ cell tiếp theo (data vẫn ở)
3. Hoặc chạy 1 domain/lần thay vì tất cả cùng lúc

**Nếu muốn tăng accuracy:**
- Tăng EPOCHS từ 10 → 20 (nhưng có rủi ro timeout)
- Chạy trên Colab Pro (T4 unlimited hoặc V100)

**Cách lấy results:**
- Download .tar.gz → extract → xem `experiments/results_pair/*/results.json`
- Hoặc print summary từ cell trước khi download
